# Interpreter les petits runs ViT

Ce notebook lit les artefacts des trois approches du projet : classification supervisee, masked autoencoder (MAE) et DINO teacher/student.

Il detecte automatiquement les derniers runs sous `project_root/runs/<approach>` (structure attendue : `history.json`, `metrics.json`, checkpoints, figures). On ne relance rien ici : on interprete les courbes, metriques et figures deja sauvegardees. Les courbes viennent de `history.json` (vraies valeurs epoch par epoch). Quand aucun checkpoint "best" n'a ete sauvegarde, le notebook infere le meilleur epoch depuis les metriques de validation.

Pour les prochains runs qualitatifs, privilegiez STL-10 (`configs/training/*_stl10.yaml`) : les images 96x96 rendent les reconstructions MAE et les cartes DINO plus lisibles. Tiny-ImageNet (`configs/training/*_tiny_imagenet.yaml`) convient aussi pour des runs plus longs avec un espace de classes plus large.

---

| | |
|---|---|
| **Approches** | Classification supervisee, MAE (He et al. 2021), DINO (Caron et al. 2021) |
| **Datasets** | CIFAR-10, STL-10, TinyImageNet |
| **Sujet** | Lecture et interpretation des artefacts d'entrainement |

Ce notebook ne relance aucun entrainement : il lit les artefacts (metriques, courbes, figures, checkpoints) produits par les runs deja effectues, et propose un cadre d'interpretation honnete. Que les `loss` descendent ou que les accuracy montent, rien n'est acquis tant que les courbes de validation, les figures qualitatives et les ecarts train/val n'ont pas ete examines.

L'objectif n'est pas de battre un benchmark. C'est de comprendre **ce que chaque approche apprend vraiment** a petite echelle, de detecter les pathologies courantes (surapprentissage, effondrement de representation, reconstruction sans semantique), et de formuler un jugement argumente sur les resultats observes.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

APPROACHES = ("classification", "mae", "dino")
PRIORITY_FIGURES = {
    "classification": [
        "training_curves.png",
        "confusion_matrix.png",
        "external_predictions.png",
        "predictions.png",
        "attention_map.png",
    ],
    "mae": [
        "training_curves.png",
        "external_reconstruction.png",
        "reconstruction.png",
        "reconstruction_errors.png",
        "nearest_neighbors.png",
    ],
    "dino": [
        "training_curves.png",
        "external_dino_attention.png",
        "nearest_neighbors.png",
        "dino_attention_diagnostics.png",
        "attention_map.png",
    ],
}
BEST_HISTORY_RULES = {
    "classification": [("val_accuracy", "max"), ("val_loss", "min")],
    "mae": [("val_loss", "min")],
    "dino": [("val_loss", "min")],
}


def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find pyproject.toml when walking upward from {start}"
    )


def find_runs(project_root, approach):
    project_root = Path(project_root)
    search_roots = [
        project_root / "runs" / approach,
        project_root / "src" / "vit_from_scratch" / "runs" / approach,
    ]

    run_dirs = []
    seen = set()
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for candidate in sorted(search_root.iterdir()):
            if not candidate.is_dir():
                continue
            resolved = candidate.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            run_dirs.append(candidate)
    return run_dirs


def latest_run(project_root, approach):
    candidates = [
        run_dir for run_dir in find_runs(project_root, approach)
        if (run_dir / "history.json").exists()
    ]
    if not candidates:
        return None
    return max(
        candidates,
        key=lambda run_dir: ((run_dir / "history.json").stat().st_mtime, run_dir.name),
    )


def read_json_dict(path):
    path = Path(path)
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    return payload if isinstance(payload, dict) else None


def relative_path(path):
    if path is None:
        return None
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def latest_metric_values(history):
    if not isinstance(history, dict):
        return {}
    return {
        key: values[-1]
        for key, values in history.items()
        if isinstance(values, list) and values
    }


def flatten_scalar_metrics(payload, prefix=""):
    if not isinstance(payload, dict):
        return {}

    flattened = {}
    for key, value in payload.items():
        full_key = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metrics(value, prefix=full_key))
        elif isinstance(value, (int, float, str)) and not isinstance(value, bool):
            flattened[full_key] = value
    return flattened


def extract_metric_section(metrics_blob, approach):
    if not isinstance(metrics_blob, dict):
        return {}
    section = metrics_blob.get(approach, metrics_blob)
    return section if isinstance(section, dict) else {}


def ordered_figures(run_dir, approach):
    figure_dir = Path(run_dir) / "figures"
    if not figure_dir.exists():
        return []

    priority = {name: index for index, name in enumerate(PRIORITY_FIGURES.get(approach, []))}
    figure_paths = sorted(figure_dir.glob("*.png"))
    return sorted(
        figure_paths,
        key=lambda path: (priority.get(path.name, len(priority)), path.name),
    )


def checkpoint_epoch_map(checkpoints):
    epoch_to_name = {}
    for checkpoint_path in checkpoints:
        stem = Path(checkpoint_path).stem
        if "_epoch_" not in stem:
            continue
        _, epoch_text = stem.rsplit("_epoch_", 1)
        if epoch_text.isdigit():
            epoch_to_name[int(epoch_text)] = Path(checkpoint_path).name
    return epoch_to_name


def best_history_points(history, approach, checkpoints):
    if not isinstance(history, dict):
        return []

    checkpoint_by_epoch = checkpoint_epoch_map(checkpoints)
    best_points = []
    for metric_name, direction in BEST_HISTORY_RULES.get(approach, []):
        values = history.get(metric_name, [])
        if not isinstance(values, list) or not values:
            continue

        numeric_points = [
            (epoch, value)
            for epoch, value in enumerate(values, start=1)
            if isinstance(value, (int, float)) and not isinstance(value, bool)
        ]
        if not numeric_points:
            continue

        if direction == "max":
            best_epoch, best_value = max(numeric_points, key=lambda item: (item[1], item[0]))
        else:
            best_epoch, best_value = min(numeric_points, key=lambda item: (item[1], item[0]))

        best_points.append(
            {
                "metric": metric_name,
                "direction": direction,
                "epoch": best_epoch,
                "value": best_value,
                "checkpoint": checkpoint_by_epoch.get(best_epoch),
            }
        )
    return best_points


def extract_best_metric_metadata(scalar_metrics):
    return {
        key: value
        for key, value in scalar_metrics.items()
        if "best" in key.lower()
    }


def summarize_run(run_dir, approach):
    if run_dir is None:
        return {
            "run_dir": None,
            "config_path": None,
            "config_payload": {},
            "resume_from": None,
            "resumed_from_epoch": None,
            "target_epochs": None,
            "history_path": None,
            "metrics_path": None,
            "history": {},
            "metrics_blob": None,
            "metric_section": {},
            "latest_metrics": {},
            "best_metric_metadata": {},
            "best_history": [],
            "checkpoints": [],
            "figures": [],
            "metrics_message": "No run detected.",
        }

    config_path = Path(run_dir) / "config.json"
    history_path = Path(run_dir) / "history.json"
    metrics_path = Path(run_dir) / "metrics.json"
    config_payload = read_json_dict(config_path) or {}
    history = read_json_dict(history_path) or {}
    metrics_blob = read_json_dict(metrics_path)
    metric_section = extract_metric_section(metrics_blob, approach)
    metric_source = metric_section or (metrics_blob if isinstance(metrics_blob, dict) else {})
    resume_from = config_payload.get("resume_from")
    if isinstance(resume_from, str) and resume_from:
        resume_from = relative_path(Path(resume_from))
    else:
        resume_from = None
    resumed_from_epoch = config_payload.get("resumed_from_epoch")
    target_epochs = config_payload.get("target_epochs", config_payload.get("epochs"))

    if metrics_blob is None:
        metrics_message = (
            f"{relative_path(metrics_path)} is missing; the notebook will keep working with history.json and figures only."
        )
    elif metric_section:
        metrics_message = f"Loaded metrics from {relative_path(metrics_path)}."
    else:
        metrics_message = (
            f"Loaded {relative_path(metrics_path)}, but no dedicated '{approach}' section was found; showing the available scalar values."
        )

    checkpoints = sorted((Path(run_dir) / "checkpoints").glob("*.pt"))
    figures = ordered_figures(run_dir, approach)
    scalar_metrics = flatten_scalar_metrics(metric_source)
    best_metric_metadata = extract_best_metric_metadata(scalar_metrics)
    best_history = best_history_points(history, approach, checkpoints)
    return {
        "run_dir": relative_path(run_dir),
        "config_path": relative_path(config_path) if config_path.exists() else None,
        "config_payload": config_payload,
        "resume_from": resume_from,
        "resumed_from_epoch": resumed_from_epoch,
        "target_epochs": target_epochs,
        "history_path": relative_path(history_path),
        "metrics_path": relative_path(metrics_path) if metrics_path.exists() else None,
        "history": history,
        "metrics_blob": metrics_blob,
        "metric_section": metric_section,
        "latest_metrics": latest_metric_values(history),
        "scalar_metrics": scalar_metrics,
        "best_metric_metadata": best_metric_metadata,
        "best_history": best_history,
        "checkpoints": [path.name for path in checkpoints],
        "figures": [path.name for path in figures],
        "metrics_message": metrics_message,
    }


def last_value(mapping, key):
    values = mapping.get(key, [])
    if isinstance(values, list) and values:
        return values[-1]
    return None


def pick_metric(section, *keys):
    for key in keys:
        value = section.get(key)
        if value is not None:
            return value
    return None


def format_value(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


PROJECT_ROOT = find_project_root()
DEFAULT_RUN_DIRS = {
    approach: latest_run(PROJECT_ROOT, approach)
    for approach in APPROACHES
}

print(f"Project root: {PROJECT_ROOT}")
for approach, run_dir in DEFAULT_RUN_DIRS.items():
    print(f"{approach}: {run_dir}")


## Selectionner les runs

La cellule suivante permet de forcer un run particulier. Par defaut, elle reprend les derniers runs detectes pour `classification`, `mae` et `dino`.


In [ ]:
RUN_DIRS = DEFAULT_RUN_DIRS.copy()

# Decommentez pour forcer un run particulier.
# RUN_DIRS["classification"] = PROJECT_ROOT / "runs" / "classification" / "20260524-232050"
# RUN_DIRS["mae"] = PROJECT_ROOT / "src" / "vit_from_scratch" / "runs" / "mae" / "20260525-005701"

RUN_SUMMARY = {
    approach: summarize_run(run_dir, approach)
    for approach, run_dir in RUN_DIRS.items()
}

RUN_SUMMARY


## Grille de lecture par approche

Avant d'inspecter les metriques et figures, voici ce qu'il faut chercher pour chaque approche. Ces criteres permettent de distinguer un run instructif d'un run qu'il faudra relancer.

### Classification supervisee

- **Ecart train / val accuracy** : un grand ecart positif (`train_accuracy` >> `val_accuracy`) est le principal indicateur de surapprentissage. Verifier les augmentations, la regularisation et la taille effective du jeu de donnees.
- **Plateau de val accuracy** : si la courbe de validation se stabilise et ne progresse plus pendant plusieurs epochs, l'apprentissage a sature, inutile de continuer sans changer le LR ou la capacite du modele.
- **Cartes d'attention** (`attention_map.png`) : dans un modele bien entraine, le token CLS doit se concentrer sur la region pertinente de l'image (l'objet), pas sur le fond ou les coins. Des cartes uniformes ou aleatoires signalent un modele sous-entraine.

### MAE (Masked Autoencoder)

- **Trajectoire de la loss de reconstruction** : elle doit baisser puis se stabiliser. Une loss qui oscille indique un LR trop eleve ; une loss qui stagne des les premiers epochs suggere que le masquage est trop facile ou trop difficile.
- **Qualite visuelle des reconstructions** (`reconstruction.png`) : les patches masques doivent etre remplis de maniere coherente avec le contexte. Une reconstruction uniformement grise ou bruiteee indique que le modele n'a rien appris au-dela de la moyenne des pixels.
- **Carte d'erreur de reconstruction** (`reconstruction_errors.png`) : les erreurs residuelles doivent etre concentrees sur les contours et les details fins, pas reparties uniformement. Une erreur uniforme = lissage sans comprehension structurelle.

### DINO (Self-supervised distillation)

- **kNN accuracy** : c'est le proxy le plus direct de la qualite de la representation. Des embeddings CLS semantiquement utiles regroupent des images similaires meme sans fine-tuning. Une kNN accuracy nulle ou aleatoire indique que la representation n'a rien capture.
- **Entropy des tetes d'attention** : dans un modele converge, les tetes doivent se specialiser, certaines tetes ont une entropie basse (focus sur une region precise), d'autres plus haute (vue globale). Une entropie uniforme et elevee sur toutes les tetes = modele sous-entraine.
- **Alignement teacher/student** : la sensibilite au momentum EMA est forte. Un momentum trop faible (teacher qui bouge trop vite) destabilise l'entrainement ; trop fort, le teacher devient une moyenne paresseuse. Surveiller la convergence de la loss et la stabilite des entropies.


## Lire les metriques sans se raconter d'histoires

Une `loss` qui baisse montre que l'objectif d'entrainement s'ameliore, pas que le modele est bon. La question : **cet objectif correspond-il a la qualite voulue ?**

Rappels :

- **Classification** : cross-entropy $$L = -\sum_i y_i log p_i$$, accuracy = predictions correctes / total. Une loss basse peut coexister avec une mauvaise calibration ou un gros ecart train/val.
- **MAE** : reconstruction pixel a pixel, $$PSNR = 10 log10(MAX^2 / MSE)$$. Des pixels propres ne garantissent pas des features semantiquement utiles.
- **DINO** : la loss mesure l'accord teacher/student, pas une accuracy downstream. Surveiller l'entropy $$H(p) = -\sum_i p_i log p_i$$, la confidence, un proxy kNN/linear probe sur les embeddings CLS, et les cartes d'attention.

Quoi lire par approche :

- **Classification** : `val_accuracy`, confusion matrix, figures de predictions, ecart `train_accuracy` vs `val_accuracy`.
- **MAE** : `reconstruction_mae`, `reconstruction_mse`, `reconstruction_psnr`, puis `reconstruction.png` et `reconstruction_errors.png`. `nearest_neighbors.png` aide a juger si les embeddings capturent plus qu'un lissage local.
- **DINO** : loss = alignement interne. Preferer `student_entropy`/`teacher_entropy`, `student_confidence`/`teacher_confidence`, `knn_accuracy` sur embeddings CLS, et `nearest_neighbors.png` / `dino_attention_diagnostics.png`.

Le notebook distingue **dernier checkpoint** et **meilleur checkpoint plausible**. Sans `best_model.pt` explicite, on reconstruit le signal depuis `history.json` et on verifie si le fichier d'epoch correspondant est encore present apres rotation.


In [ ]:
for approach, summary in RUN_SUMMARY.items():
    print(f"\n=== {approach} summary ===")
    if summary["run_dir"] is None:
        print("No run detected.")
        continue

    print(f"Run: {summary['run_dir']}")
    if summary["config_path"] is not None:
        print(f"Config: {summary['config_path']}")
    print(f"History: {summary['history_path']}")
    if summary["resume_from"] is None:
        print("Resume: fresh run")
    else:
        print(f"Resume from: {summary['resume_from']}")
    if summary["resumed_from_epoch"] is not None:
        print(f"Resumed from epoch: {summary['resumed_from_epoch']}")
    if summary["target_epochs"] is not None:
        print(f"Target epochs: {summary['target_epochs']}")
    print(summary["metrics_message"])
    if summary["metrics_path"] is not None:
        print(f"Metrics path: {summary['metrics_path']}")

    metrics_payload = summary["metric_section"] or summary["metrics_blob"]
    if metrics_payload:
        print("metrics.json payload:")
        print(json.dumps(metrics_payload, indent=2, sort_keys=True))
    else:
        print("No metrics payload to display yet.")

    print("Latest values reconstructed from history.json:")
    if summary["latest_metrics"]:
        for metric_name, metric_value in summary["latest_metrics"].items():
            print(f"  - {metric_name}: {format_value(metric_value)}")
    else:
        print("  - none")

    print("Best metric metadata found in metrics.json:")
    if summary["best_metric_metadata"]:
        for metric_name, metric_value in summary["best_metric_metadata"].items():
            print(f"  - {metric_name}: {format_value(metric_value)}")
    else:
        print("  - none")

    print("Best checkpoint candidates inferred from history.json:")
    if summary["best_history"]:
        for candidate in summary["best_history"]:
            checkpoint_name = candidate["checkpoint"] or "not retained by checkpoint rotation"
            print(
                "  - "
                f"{candidate['metric']} ({candidate['direction']}) -> "
                f"epoch {candidate['epoch']} / value {format_value(candidate['value'])} / "
                f"checkpoint: {checkpoint_name}"
            )
    else:
        print("  - none")

    print("Checkpoints:")
    for checkpoint in summary["checkpoints"] or ["none"]:
        print(f"  - {checkpoint}")

    print("Figures found:")
    for figure_name in summary["figures"] or ["none"]:
        print(f"  - {figure_name}")


## Courbes et interpretation rapide

Les courbes tracent la trajectoire epoch par epoch depuis `history.json`. Un seul point = un run d'une seule epoch.

Points de vigilance :

- Une `train_loss` qui baisse peut juste refléter de la memorisation.
- En classification, `train_accuracy` peut monter pendant que `val_accuracy` baisse : signature classique de surapprentissage. Un grand ecart train/val confirme le diagnostic.
- En MAE, une `val_loss` basse = pixels proches des cibles, pas forcement des embeddings utiles en downstream.
- En DINO, la loss mesure l'accord teacher/student. Pour verifier que la representation ne s'effondre pas, croiser avec l'entropy, la confidence et un proxy kNN/linear probe.
- Pas d'early stopping automatique pour l'instant. Les configs longues utilisent un scheduler cosine + warmup : les courbes aident a voir l'effet du learning rate et a choisir l'epoch a reprendre.


In [ ]:
def plot_history(history, title):
    if not history:
        print(f"No history to plot for {title}.")
        return None

    metric_groups = [
        ("loss", "Loss"),
        ("accuracy", "Accuracy"),
        ("mask_ratio", "Mask ratio"),
        ("temperature", "Temperature"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes = list(axes)

    plotted_axes = 0
    for metric, axis_title in metric_groups:
        keys = [key for key in history if metric in key]
        if not keys:
            continue
        if plotted_axes >= len(axes):
            break
        ax = axes[plotted_axes]
        for key in keys:
            ax.plot(history[key], marker="o", label=key)
        ax.set_title(axis_title)
        ax.set_xlabel("Epoch")
        ax.legend()
        plotted_axes += 1

    for ax in axes[plotted_axes:]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def render_interpretation(approach, summary):
    history = summary["history"]
    metrics = summary["metric_section"]
    figures = set(summary["figures"])
    lines = []

    if summary["resume_from"] is None:
        lines.append("- Resume metadata: fresh run.")
    else:
        lines.append(f"- Resume metadata: resumed from `{summary['resume_from']}`.")
    if summary["resumed_from_epoch"] is not None:
        lines.append(f"- `resumed_from_epoch`: **{summary['resumed_from_epoch']}**.")
    if summary["target_epochs"] is not None:
        lines.append(f"- `target_epochs`: **{summary['target_epochs']}**.")
    if summary["best_metric_metadata"]:
        metadata_keys = ", ".join(f"`{key}`" for key in summary["best_metric_metadata"])
        lines.append(f"- `metrics.json` exposes explicit best-metric metadata via {metadata_keys}.")
    if summary["best_history"]:
        lines.append("- Best checkpoint hints were reconstructed from validation history. If a checkpoint is reported missing, checkpoint rotation already pruned that epoch file.")

    if approach == "classification":
        train_acc = last_value(history, "train_accuracy")
        val_acc = last_value(history, "val_accuracy")
        top1 = pick_metric(metrics, "top1_accuracy", "accuracy")
        lines.append("- Loss only tells you the optimizer objective improved; quality is supported mainly by validation accuracy, the confusion matrix, prediction examples, and the train/val gap.")
        lines.append("- It is perfectly possible for `train_accuracy` to keep rising while validation degrades. In that case, prefer the best validation epoch over the final checkpoint.")
        if val_acc is not None:
            lines.append(f"- Final `val_accuracy`: **{val_acc:.2%}**.")
        if top1 is not None:
            lines.append(f"- `top1_accuracy` from `metrics.json`: **{top1:.2%}**.")
        if train_acc is not None and val_acc is not None:
            gap = train_acc - val_acc
            lines.append(f"- Generalization gap (`train_accuracy - val_accuracy`): **{gap:+.2%}**. A large positive gap is a warning sign for overfitting.")
        lines.append("- Read `confusion_matrix.png` and the raw confusion matrix to see whether the model fails on one class or spreads errors everywhere.")
        if "predictions.png" in figures:
            lines.append("- `predictions.png` is available: use it to check if the right class is predicted for the right reasons, not just with high confidence.")

    elif approach == "mae":
        val_loss = last_value(history, "val_loss")
        mse = pick_metric(metrics, "reconstruction_mse")
        mae = pick_metric(metrics, "reconstruction_mae")
        psnr = pick_metric(metrics, "reconstruction_psnr")
        lines.append("- A lower reconstruction loss means the decoder matches pixels more closely, but that does not prove the latent space is semantically strong.")
        if val_loss is not None:
            lines.append(f"- Final `val_loss`: **{val_loss:.4f}**.")
        if mse is not None:
            lines.append(f"- `reconstruction_mse`: **{mse:.4f}**.")
        if mae is not None:
            lines.append(f"- `reconstruction_mae`: **{mae:.4f}**.")
        if psnr is not None:
            lines.append(f"- `reconstruction_psnr`: **{psnr:.2f} dB**. Higher PSNR usually means cleaner pixels, not automatically better semantics.")
        lines.append("- Compare `reconstruction.png` with `reconstruction_errors.png`: if errors stay small but structured objects disappear, the model may still be missing useful semantics.")
        if "nearest_neighbors.png" in figures:
            lines.append("- `nearest_neighbors.png` is available: treat it as a quick proxy for whether embeddings group semantically similar images.")

    elif approach == "dino":
        val_loss = last_value(history, "val_loss")
        student_entropy = pick_metric(metrics, "student_entropy", "entropy")
        teacher_entropy = pick_metric(metrics, "teacher_entropy")
        student_conf = pick_metric(metrics, "student_confidence", "confidence")
        teacher_conf = pick_metric(metrics, "teacher_confidence")
        knn_acc = pick_metric(metrics, "knn_accuracy")
        attention_entropy = pick_metric(metrics, "attention_entropy")
        attention_peak_mass = pick_metric(metrics, "attention_peak_mass")
        lines.append("- A low DINO loss means student and teacher agree more; it is not a downstream accuracy metric.")
        if val_loss is not None:
            lines.append(f"- Final `val_loss`: **{val_loss:.4f}**. Read it as agreement strength, not task performance.")
        if student_entropy is not None or teacher_entropy is not None:
            lines.append("- Entropy diagnostics help detect collapse: very low entropy together with extreme confidence can mean the model predicts nearly the same thing for everything.")
        if student_entropy is not None:
            lines.append(f"- `student_entropy`: **{student_entropy:.4f}**.")
        if teacher_entropy is not None:
            lines.append(f"- `teacher_entropy`: **{teacher_entropy:.4f}**.")
        if student_conf is not None:
            lines.append(f"- `student_confidence`: **{student_conf:.4f}**.")
        if teacher_conf is not None:
            lines.append(f"- `teacher_confidence`: **{teacher_conf:.4f}**.")
        if knn_acc is not None:
            lines.append(f"- `knn_accuracy` on CLS embeddings: **{knn_acc:.2%}**. This is a cheap proxy for representation quality before a full linear probe.")
        else:
            lines.append("- If `knn_accuracy` or a linear probe is absent, treat the run as only partially evaluated. DINO needs a representation-level check.")
        if attention_entropy is not None:
            lines.append(f"- `attention_entropy`: **{attention_entropy:.4f}**.")
        if attention_peak_mass is not None:
            lines.append(f"- `attention_peak_mass`: **{attention_peak_mass:.4f}**. A very peaky map can be good object focus or a collapse symptom depending on the visualizations.")
        lines.append("- Use `nearest_neighbors.png` and `dino_attention_diagnostics.png` in the spirit of the DINO paper: do neighbors stay semantically coherent, and do attention maps discover objects rather than background noise?")

    return "\n".join(lines)


for approach, summary in RUN_SUMMARY.items():
    print(f"\n=== {approach} ===")
    if summary["run_dir"] is None:
        print("No run detected.")
        continue

    fig = plot_history(summary["history"], f"{approach}: {Path(summary['run_dir']).name}")
    if fig is not None:
        plt.show()

    display(Markdown(f"### {approach} interpretation\n{render_interpretation(approach, summary)}"))


### Comment lire les courbes d'entrainement

Les courbes ci-dessus representent la trajectoire de l'optimisation epoch par epoch. Voici les signaux a rechercher :

- **Courbe lisse et monotone** : indique un entrainement stable. Les oscillations marquees suggerent un learning rate trop eleve, le modele saute par-dessus les minima sans s'y installer.
- **Divergence train / val** : si `train_loss` descend mais `val_loss` remonte (ou si `train_accuracy` monte pendant que `val_accuracy` plafonne), c'est la signature du surapprentissage. Les pistes de correction : augmentation de donnees plus agressive, dropout, weight decay, ou simplement plus de donnees.
- **MAE loss proche de 1.0** : une loss de reconstruction qui stagne autour de 1.0 indique que le modele predit essentiellement la valeur moyenne des pixels masques, sans exploiter le contexte local. Il n'a quasiment rien appris. C'est souvent un probleme de LR, de taux de masquage, ou d'un run trop court.
- **Plateau premature** : si la loss ne bouge plus des les premieres epochs, verifier le scheduler (warmup trop court, LR initial trop faible) et s'assurer que les donnees sont bien melangees.


## Images externes reelles

Les figures doivent etre inspectees sur de vraies images, pas du bruit. Placez des `.jpg`/`.jpeg`/`.png` sous `data/external_images/`, puis relancez un run avec `--external-image-dir data/external_images/`.

Pour remplir ce dossier :

```bash
python scripts/download_external_images.py
```

Les figures `external_predictions.png`, `external_reconstruction.png` ou `external_dino_attention.png` apparaissent alors dans `runs/<approach>/<run>/figures/`. Les configs STL-10 pointent deja vers ce dossier ; s'il est absent ou vide, le CLI saute les figures externes.


In [ ]:
EXTERNAL_IMAGE_DIR = PROJECT_ROOT / 'data' / 'external_images'
external_images = []
if EXTERNAL_IMAGE_DIR.exists():
    external_images = sorted(
        path for path in EXTERNAL_IMAGE_DIR.iterdir()
        if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    )

if not external_images:
    print('No external images found yet. Run: python scripts/download_external_images.py')
else:
    print(f'External images in {EXTERNAL_IMAGE_DIR}:')
    for image_path in external_images[:8]:
        print(f'- {image_path.name}')
        display(Image(filename=str(image_path), width=220))


## Figures de diagnostic

Les figures apportent un signal qualitatif la ou les scalaires ne suffisent pas. Ordre de priorite :

- **classification** : `confusion_matrix.png`, `predictions.png`, `attention_map.png`
- **MAE** : `reconstruction.png`, `reconstruction_errors.png`, `nearest_neighbors.png`
- **DINO** : `nearest_neighbors.png`, `dino_attention_diagnostics.png`, `attention_map.png`

Sans `metrics.json`, on peut deja inspecter `history.json` et les figures sauvees.


### Comment lire les figures de diagnostic

Les figures qualitatives completent les scalaires. Voici les criteres d'interpretation :

- **Cartes d'attention** : dans un modele suffisamment entraine, les tetes doivent se specialiser. Certaines tetes vont isoler le sujet principal, d'autres les contours, d'autres les zones de texture. Des cartes qui ressemblent a du bruit aleatoire (sans structure ni focus) signalent un modele sous-entraine ou un effondrement de la representation. Ce n'est pas un echec en soi : c'est un diagnostic.
- **Reconstructions MAE** : une reconstruction floue signifie que le modele a capture la structure globale (couleurs dominantes, contours grossiers) mais pas les details fins. C'est attendu pour un petit ViT entraine peu d'epochs. Une reconstruction nette avec preservation des textures indique que l'encodeur a appris des features de bas niveau pertinentes, mais cela ne garantit pas la qualite des embeddings en downstream. Les deux qualites sont independantes.
- **kNN accuracy (DINO)** : entre 40 % et 60 % sur STL-10 avec un petit ViT entraine de zero est un resultat acceptable, cela signifie que la representation contient de l'information semantique non triviale. Au-dela de 70 %, il faudrait soit un modele plus grand, soit plus de donnees, soit un pre-entrainement ImageNet. Ne pas interpreter une kNN accuracy modeste comme un echec : c'est une indication de la capacite du regime d'entrainement, pas une limite du projet.


In [ ]:
for approach, run_dir in RUN_DIRS.items():
    print(f"\n=== {approach} figures ===")
    if run_dir is None:
        print("No run detected.")
        continue

    figure_paths = ordered_figures(run_dir, approach)
    if not figure_paths:
        print(f"No PNG figures found in {Path(run_dir) / 'figures'}")
        continue

    for figure_path in figure_paths:
        print(figure_path.name)
        display(Image(filename=str(figure_path)))


## Conclusion

### Ce que ces runs nous ont appris

Les trois approches (classification supervisee, MAE et DINO) couvrent trois regimes d'apprentissage visuel : supervision directe, reconstruction de signal et distillation auto-supervisee. A petite echelle, leurs forces et limites sont visibles et instructives.

La classification supervisee donne des signaux clairs (accuracy, matrice de confusion), mais reste fragile au surapprentissage des que les donnees sont insuffisantes ou l'augmentation trop faible. Le MAE apprend a reconstruire des structures visuelles coherentes, mais la qualite pixel et la qualite semantique des embeddings sont deux choses distinctes, il faut les evaluer separement. DINO produit des representations zero-shot dont la qualite se lit dans les cartes d'attention et la kNN accuracy : un run qui converge sans effondrement, avec des tetes d'attention specialisees, est un bon signe.

### Evaluation honnete

Ces experiences sont des experiences a petite echelle, pas des benchmarks. Les comparaisons directes avec les papiers originaux (He et al. 2021, Caron et al. 2021) n'ont pas de sens : les ressources de calcul, la taille des datasets et la duree d'entrainement sont d'ordres de grandeur differents. Le but ici est de reimplementer les mecanismes cles, de verifier qu'ils se comportent comme prevu, et d'interpreter les signaux honnetement.

Les courbes et les figures sont le livrable principal, pas les chiffres d'accuracy.

### Ce qui ameliorerait les resultats

Dans l'ordre de priorite :

1. **Plus de donnees** : CIFAR-10 (50k images) est serree pour DINO et MAE. STL-10 ou TinyImageNet offrent plus de diversite visuelle. ImageNet (meme un sous-ensemble) changerait l'ordre de grandeur.
2. **Entrainement plus long** : MAE et DINO necessitent typiquement 200–800 epochs pour produire des representations stables. Les runs courts donnent des signaux utiles mais pas des representations exploitables en downstream.
3. **Modele plus grand** : un ViT-S ou ViT-B avec patch size 8 (au lieu de 16) capture plus de details locaux, particulierement important pour MAE et les cartes d'attention DINO.
4. **Pre-entrainement ImageNet** : initialiser depuis des poids publics (DeiT, MAE official, DINO official) puis fine-tuner sur le dataset cible est la voie la plus pragmatique pour atteindre des performances publishables.
5. **Augmentation adaptee** : pour DINO, les multi-crop views avec des resolutions mixtes sont critiques. Pour la classification, mixup et cutmix reduisent significativement le surapprentissage.

Les notebooks, les courbes d'entrainement et les figures de diagnostic constituent la contribution de ce projet. Ils documentent une reimplementation soigneuse des trois approches et montrent que les mecanismes fondamentaux fonctionnent a l'echelle accessible.
